## `build:` vs `image:`

Every service has to say *what* to run, with one of two keys — pull a ready-made image, or build one from a Dockerfile.

**`image:`** runs a pre-built image, pulled from a registry if it's not local. This is what you use for off-the-shelf dependencies:

```yaml
services:
  cache:
    image: redis:7-alpine
```

**`build:`** builds an image from a Dockerfile in your project — for the code *you* wrote. The short form points at a context directory:

```yaml
services:
  web:
    build: .        # context = ., Dockerfile = ./Dockerfile
```

The long form gives you the module-02 build controls right in the YAML — a subdirectory context, a named Dockerfile, build args, and a multi-stage **`target`**:

```yaml
services:
  web:
    build:
      context: ./web
      dockerfile: Dockerfile.prod
      args:
        APP_VERSION: 1.2.3
      target: production
    image: myorg/web:1.2.3     # tag the built image with this name
```

You can give a service **both `build:` and `image:`** — Compose builds, then **tags** the result with `image:`, and that tag is what `docker compose push` would publish.

The behaviour that trips people up: **`docker compose up` only builds an image if it doesn't already exist locally.** Change your Dockerfile or source, run `up` again, and Compose happily reuses the stale image. To force a rebuild, use **`docker compose up --build`** (or an explicit `docker compose build`). A good habit during active development is `up --build`; in CI, an explicit `build` step before `up`.